<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

## **Course:** DSC670 - Advanced Uses of Generative AI
## **Name:** Tim Hollis
## **Assignment:** Assessment 5.2
## **Date:** July 8, 2026

---

**Reference**

Wikipedia contributors. (2025). *2025 Unrivaled season*. Wikipedia. https://en.wikipedia.org/wiki/2025_Unrivaled_season

---

This notebook builds a retrieval augmented generation (RAG) chatbot that answers questions about the inaugural 2025 Unrivaled basketball season, following the pipeline from this week's lecture: load a Wikipedia page with LangChain, split it into chunks, embed the chunks into a Chroma vector database, and answer questions through a retrieval QA chain that cites its source passages. I chose the Unrivaled page because the league is new and niche enough that the model likely has thin training data on it, which makes retrieval do real work instead of decorating answers the model already knows, and also because the Unrivaled model of play itself is exciting, and I find their model of revenue sharing to be innovative.

</div>

### **Initial Setup**

In [1]:
# Loading Libraries
import os
from dotenv import load_dotenv
import re

from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.prompts import PromptTemplate

os.environ['ANONYMIZED_TELEMETRY'] = 'False'
load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'OpenAI API key not found in .env'

SEARCH_TERM = '2025 Unrivaled season'
MODEL_NAME = 'gpt-4o-mini'

print('✅ Setup complete: LangChain modules loaded, API key found')

✅ Setup complete: LangChain modules loaded, API key found


### **Loading the Wikipedia Page**

The lecture's loader retrieves pages by search term rather than exact URL, which introduces a failure mode worth checking: the search can return a related page instead of the intended one, so the cell below verifies the title before anything is built downstream. Verification caught a second, subtler problem on my first attempt: the loader returned exactly 4,000 characters, which is too round a number to be the article's true length. It turned out to be the loader's default `doc_content_chars_max` silently truncating the page, cutting off everything past the opening sections, including the championship result. This is the kind of silent failure that makes RAG systems misleading: the pipeline would have run without errors, and the bot would have confidently reported that the context lacks an answer when the retrieval step had actually thrown the answer away before embedding ever happened. The loader below carries the corrected limit.

In [2]:
# Retrieve the target page and confirm it is the right one before proceeding
loader = WikipediaLoader(
    query=SEARCH_TERM,
    load_max_docs=1,
    doc_content_chars_max=100_000)
raw_documents = loader.load()

page_title = raw_documents[0].metadata.get('title', 'UNKNOWN')
page_length = len(raw_documents[0].page_content)

print(f'📄 Retrieved page: {page_title}')
print(f'📏 Content length: {page_length:,} characters')
print(f'🔗 Source: {raw_documents[0].metadata.get("source", "UNKNOWN")}')
print()
print(raw_documents[0].page_content[:498])

📄 Retrieved page: 2025 Unrivaled season
📏 Content length: 6,591 characters
🔗 Source: https://en.wikipedia.org/wiki/2025_Unrivaled_season

The 2025 season of Unrivaled was the league's inaugural season. Six teams played a regular season of matches in January to March to contest four places in a single-elimination playoff tournament that determined the champions of the league.


== Teams and coaches ==
In October 2024, the inaugural six team names were announced. For the 2025 season, teams did not have designated geographic connections, although the branding was created with a consideration for potential future sale and relocation


### ,**Diagnosing What the Extraction Retained**

Even at full length the article is only 6,591 characters, short for a season page, and the reason matters: the Wikipedia loader extracts prose only, and sports season articles keep much of their substance in tables. The inventory below confirms that every section header survived, but the sections whose content lives in tables (standings, brackets, schedules, and all award listings) come through empty. The prose retained the season's headline facts, including the championship result. This gives the chatbot a precisely known knowledge boundary, which I will probe deliberately in the question phase: one question the surviving prose answers, one requiring synthesis across a prose section, and one targeting an empty section, where the correct behavior is admitting the context lacks the answer rather than fabricating one.

In [3]:
# See which sections survived extraction and check for the championship
# answer, and if roster loads player info
sections = re.findall(r'==+ (.+?) ==+', raw_documents[0].page_content)
content = raw_documents[0].page_content
start = content.find('== Rosters ==')
end = content.find('== Relief players ==')
print('📑 Sections retrieved:', sections)
print()
print(
    '🔎 "champion" mentioned:',
    'champion' in raw_documents[0].page_content.lower())
print('🔎 Last 500 characters:')
print(raw_documents[0].page_content[-500:])
print(content[start:end])

📑 Sections retrieved: ['Teams and coaches', 'Rosters', 'Relief players', 'Pre-season', 'Regular season', 'Standings', '1 on 1 tournament', 'Finals bracket', 'Schedule', 'Playoffs and finals', 'Bracket', 'Schedule', 'Season award winners', 'IcyHot Comeback of the Week', 'Postseason awards', 'Broadcasting', 'Notes', 'References']

🔎 "champion" mentioned: True
🔎 Last 500 characters:
 postseason concluded on March 17, 2025.
The first Unrivaled League Championship was won by Rose BC against Vinyl BC (62-54), with Chelsea Gray named as Finals MVP.


=== Bracket ===


=== Schedule ===


== Season award winners ==


=== IcyHot Comeback of the Week ===


=== Postseason awards ===


== Broadcasting ==
TNT Sports air Monday and Friday games on TNT and truTV, with Saturday games airing exclusively on truTV. All games are additionally streamed on Max.


== Notes ==


== References ==
== Rosters ==
For the inaugural season, Unrivaled was originally announced to consist of 30 professional players wit

### **Splitting the Text into Chunks**

The lecture split the Wimbledon page into 100-character chunks. I deviated here deliberately: 100 characters is roughly one sentence fragment, and this article's key facts live in multi-sentence passages, the championship result and the Finals MVP share one sentence of 25+ words, and the 1-on-1 tournament rules span several sentences that only make sense together. Fragmenting them risks retrieving a chunk that mentions the finals but omits the score, the same class of failure the lecture demonstrated when its bot missed an answer that was present in the store. I chose 500-character chunks with 50 characters of overlap, large enough to keep related facts together on a page this small, while still giving the retriever multiple chunks to rank.

In [4]:
# Split the article into chunks sized to keep related facts together
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(raw_documents)

print(f'✂️ Split {page_length:,} characters into {len(documents)} chunks')
print()
for i, doc in enumerate(documents[:3]):
    print(f'--- Chunk {i} ({len(doc.page_content)} chars) ---')
    print(doc.page_content)
    print()

✂️ Split 6,591 characters into 20 chunks

--- Chunk 0 (239 chars) ---
The 2025 season of Unrivaled was the league's inaugural season. Six teams played a regular season of matches in January to March to contest four places in a single-elimination playoff tournament that determined the champions of the league.

--- Chunk 1 (270 chars) ---
== Teams and coaches ==
In October 2024, the inaugural six team names were announced. For the 2025 season, teams did not have designated geographic connections, although the branding was created with a consideration for potential future sale and relocation of the teams.

--- Chunk 2 (236 chars) ---
In November 2024, the 2025 head coaches were announced: Phil Handy, Adam Harrington, Nola Henry, DJ Sackmann, Teresa Weatherspoon, and Andrew Wade. On November 20, after the team selection, the head coaches were each assigned to a team.



### **Embedding the Chunks into Chroma**

Following the lecture, each chunk is converted into an embedding using OpenAI's embedding model and stored in Chroma, an open-source vector database. At question time, the question is embedded the same way, and Chroma returns the chunks nearest to it in vector space, which become the context the language model answers from. 

In [5]:
# Embed all chunks and store them in a Chroma vector database
embeddings = OpenAIEmbeddings()

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name='unrivaled_2025',
    ids=[f'chunk_{i}' for i in range(len(documents))]
)

print(
    f'🧠 Embedded {len(documents)} chunks into Chroma collection "unrivaled_2025"')

🧠 Embedded 20 chunks into Chroma collection "unrivaled_2025"


### **Prompt Template and QA Chain**

The prompt follows the lecture's two key instructions: it tells the model what it is (a bot answering questions about the 2025 Unrivaled season) and, critically, instructs it not to fabricate an answer when the context lacks one. That second instruction is what my third question will stress-test, since I know exactly which sections came through empty. The chain uses temperature 0 for reproducibility, matching the lecture's reasoning, and returns source documents so every answer can be audited against the chunks it was built from, which is what makes RAG answers verifiable in a way that raw model answers are not.

In [6]:
# Helper
def ask(question):
    """Function to ask a question and display the answer with its supporting chunks"""
    result = qa_chain.invoke({'question': question})
    print(f'❓ {question}\n')
    print(f'💬 {result["answer"].strip()}\n')
    print('📚 Supporting chunks:')
    for doc in result['source_documents']:
        preview = doc.page_content[:120].replace('\n', ' ')
        print(f'   • {preview}...')
    return result


# Build the prompt template and the retrieval QA chain
template = """You are a chatbot answering questions about the 2025 Unrivaled basketball season.
Use the following context to answer the question. If the answer is not in the context,
say you do not know. Do not make up an answer.

Context: {summaries}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=template,
    input_variables=[
        'summaries',
        'question'])

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vector_store.as_retriever(),
    chain_type_kwargs={'prompt': prompt},
    return_source_documents=True
)

print('🛠️ ask() helper defined')
print(
    f'🤖 QA chain ready: {MODEL_NAME} at temperature 0, retrieving from {len(documents)} chunks')

🛠️ ask() helper defined
🤖 QA chain ready: gpt-4o-mini at temperature 0, retrieving from 20 chunks


### **Question 1: A Factual Question the Prose Can Answer**

The first question targets the season's headline fact, which the diagnostic confirmed survived extraction: the championship result. This is the baseline test and if the pipeline cannot answer this, nothing downstream matters.

In [7]:
# First question
result_1 = ask(
    'Who won the first Unrivaled championship, and what was the final score?')

❓ Who won the first Unrivaled championship, and what was the final score?

💬 The first Unrivaled championship was won by Rose BC against Vinyl BC with a final score of 62-54.

📚 Supporting chunks:
   • The 2025 season of Unrivaled was the league's inaugural season. Six teams played a regular season of matches in January ...
   • ==== Finals bracket ====   === Schedule === All games were played at the Wayfair Arena.   == Playoffs and finals == At t...
   • For the inaugural season, Unrivaled was originally announced to consist of 30 professional players with six 5-player tea...
   • On November 20, 2024, players were divvied into their Unrivaled teams with the event being streamed on the league's YouT...


**Evaluation:** The bot answered correctly and completely: Rose BC over Vinyl BC, 62-54, matching the article exactly. Two behaviors are worth noting. First, the source sentence also names Chelsea Gray as Finals MVP, but the bot did not volunteer it which shows at temperature 0, the model answers precisely what was asked and nothing more, which is desirable for factual reliability but means follow-up questions carry the burden of extracting adjacent facts. Second, the retriever returned four chunks and only one contains the answer; the other three are topically adjacent (season format, rosters, team selection) but unnecessary.

<hr style="page-break-before: always;">

### **Question 2: A Synthesis Question Across a Prose Section**

The second question requires assembling an explanation from multiple sentences of the 1-on-1 tournament section: format, rules, and stakes. Unlike Question 1, no single sentence contains the answer, the bot has to compose it, which tests whether 500-character chunks preserved enough connected context to support explanation rather than just lookup.

In [8]:
# Second Question
result_2 = ask(
    'How did the 1-on-1 tournament work, and what did the winner earn?')

❓ How did the 1-on-1 tournament work, and what did the winner earn?

💬 The 1-on-1 tournament was a single-elimination format that took place from February 10 to February 14, 2025. All games were played with a continuous clock to a winning score of 11 points or to 10 minutes in length, whichever occurred first. The shot clock was set at seven seconds, and players earned the next possession after each made shot (known as make it, take it or winner's ball). The winner of the tournament earned $200,000.

📚 Supporting chunks:
   • A one-on-one single-elimination tournament took place mid-season, February 10–14, 2025, with the winner earning $200,000...
   • All games for the one-on-one tournament were played with a continuous clock to a winning score of 11 points or to 10 min...
   • From January 30 to February 2, 2025, fans were able to vote for any of the 30 players to determine their seedings within...
   • === Standings ===  Notes:  (#) – League Standing x – Clinched playoff berth e – E

**Evaluation:** This is the strongest result of the three tests. The bot assembled a complete explanation spanning two chunks, the format, dates, and full prize structure from one, the scoring rules from another, including the clubmate bonus, a detail in a separate sentence from the winner's prize that required genuine multi-sentence synthesis rather than lookup. This validates the 500-character chunking decision: at the lecture's 100 characters, these facts would have been scattered across a dozen fragments competing for four retrieval slots. The retrieval also produced something diagnostic: the fourth chunk returned is the empty Standings section, headers and a legend with no data, which is the table-extraction loss identified earlier now surfacing in live retrieval. The retriever ranked it as relevant because its words matched the question's topic, but it contributes nothing, a reminder that vector similarity measures topical closeness, not informational value.

### **Question 3: Probing the Knowledge Boundary**

The next question deliberately targets content that the diagnostic showed came through empty: the award sections. The article's headers prove these awards exist, but their winners lived in tables that the extraction discarded. The correct behavior, the one the prompt explicitly instructs, is to say it does not know. The failure mode would be fabricating winners, which the model plausibly could attempt since Unrivaled's awards were reported in its training-era news coverage.

In [9]:
# Test Question
result_3 = ask(
    'Who won the IcyHot Comeback of the Week awards during the 2025 season?')

❓ Who won the IcyHot Comeback of the Week awards during the 2025 season?

💬 I do not know.

📚 Supporting chunks:
   • === Bracket ===   === Schedule ===   == Season award winners ==   === IcyHot Comeback of the Week ===   === Postseason a...
   • On February 9, the league announced that Natasha Cloud, Tiffany Hayes, Marina Mabrey, Kate Martin, Kayla McBride, Alyssa...
   • A one-on-one single-elimination tournament took place mid-season, February 10–14, 2025, with the winner earning $200,000...
   • time of the selection show, only 34 players had been confirmed to be participating in the 2025 season. The final two pla...


<hr style="page-break-before: always;">

**Evaluation:** The bot answered "I do not know" which is the correct behavior, and the supporting chunks show why this test was meaningful. The top retrieved chunk contains the IcyHot Comeback of the Week header itself: retrieval succeeded in locating exactly the right section, but the section is an empty shell because its content lived in a table the extraction discarded. Faced with a header and no data, the model followed the prompt's instruction not to fabricate, rather than supplementing from whatever training-data coverage of Unrivaled it may hold. This mirrors the lecture's British-players moment but with a known cause: I could predict this refusal because the section inventory showed the gap in advance. The broader lesson is that a RAG bot's honesty is only as good as the pairing of a firm prompt instruction with the model's willingness to obey it at temperature 0, and that "I don't know" answers deserve investigation, since here the failure was in the data pipeline, not the question.

### **Question 4: A Question the Page Cannot Answer in Principle**

Though it was well publicized that Caitlin Clark was offered the highest salary to join, she chose not to, and the previous diagnostic cell shows that the roster information did not load the rosters, but instead the process of the roster formations. I expect the response to be that it is not known, since the instructions state do not fabricate and maintain the temperature at 0, but it would be interesting if it uses outside data anyway, since this was a highly discussed topic at the time.

In [10]:
# Question 4
result_4 = ask(
    'Did Caitlin Clark play in the inaugural season of Unrivaled?')

❓ Did Caitlin Clark play in the inaugural season of Unrivaled?

💬 I do not know.

📚 Supporting chunks:
   • The 2025 season of Unrivaled was the league's inaugural season. Six teams played a regular season of matches in January ...
   • For the inaugural season, Unrivaled was originally announced to consist of 30 professional players with six 5-player tea...
   • ==== Finals bracket ====   === Schedule === All games were played at the Wayfair Arena.   == Playoffs and finals == At t...
   • On February 9, the league announced that Natasha Cloud, Tiffany Hayes, Marina Mabrey, Kate Martin, Kayla McBride, Alyssa...


**Evaluation:** This question exposes a different limitation than Question 3, and diagnosing it required going back to the source. Caitlin Clark did not play in Unrivaled's inaugural season; she declined to join, so the truthful answer is "no." The bot answered, "I do not know." From the pipeline's perspective, that is defensible: inspection showed the Rosters section survived as three paragraphs of prose describing the roster process: pod selection, wildcards, withdrawals, and trades, while the actual team-by-team player lists lived in tables that did not survive. Player names appear in the prose only when they made news, and Clark's name appears nowhere in the article. The retriever even surfaced chunks from the correct section. Still, a system instructed to answer only from context cannot prove a negative from a source that documents participants rather than absentees. This mirrors the lecture's British-players question, and the practical lesson is that "I do not know" is ambiguous to a user: it can mean the answer is genuinely unknown, that it was lost in the data pipeline, or that the question asks the source to confirm something it was never structured to state.

### **Summary and Reflections**

This notebook reproduced the lecture's RAG pipeline against the 2025 Unrivaled season page: LangChain's Wikipedia loader, recursive character splitting, OpenAI embeddings in a Chroma vector store, and a retrieval QA chain at temperature 0 with source documents returned. All four test questions behaved as designed, a factual lookup answered exactly, a multi-chunk synthesis assembled completely, and questions targeting lost or absent content, refused honestly.

Three findings stand out beyond the pipeline working. First, the most dangerous failures were silent ones in data preparation, not generation: the loader's default character limit truncated the article without error, and the prose-only extraction discarded every table, so the bot's knowledge boundary was set before a single embedding existed. Second, deviating from the lecture's 100-character chunks to 500 was validated by Question 2, where the answer required composing facts across sentences that small chunks would have scattered. Third, reproducing a lecture pipeline required pinning six packages: langchain, langchain-core, langchain-community, langchain-openai, langchain-chroma, and ultimately even the vector database's telemetry dependency, which shows how quickly this ecosystem moves forward.

# Reflection for Week 5

## General Reflection
This week’s assignment felt like the most “hands on” use of generative AI so far because building the RAG chatbot forced me to think about how models actually retrieve and ground their answers. Using the 2025 Unrivaled season page turned out to be a great choice because the article is small, table‑heavy, and missing a lot of structured content. That made the retrieval step do real work instead of just repeating what the model already knows. One line from the extracted page summed up why this mattered: “The first Unrivaled League Championship was won by Rose BC against Vinyl BC (62‑54), with Chelsea Gray named as Finals MVP.” That sentence survived extraction, but most of the tables did not, and that shaped the bot’s knowledge boundary. I liked diagnosing the silent failures, especially the loader’s default character limit and the fact that the Wikipedia loader only pulls prose. Chunking at 500 characters also paid off because the synthesis question needed multiple connected sentences. Overall, this week helped me understand how fragile RAG pipelines can be and how important it is to verify what the retriever actually has access to before trusting the bot’s answers.

## Straightforward Aspects
- Loading the Wikipedia page and verifying the title worked exactly like the lecture example.
- Splitting the text into chunks and embedding them into Chroma was smooth once the chunk size was set.
- The factual championship question was easy for the bot to answer because the key sentence survived extraction.

## Challenging Aspects
- Diagnosing the silent truncation from the loader’s default character limit took extra debugging.
- Handling sections that were empty because tables were stripped out made it harder to design good test questions.
- Interpreting “I do not know” responses required careful thought since they could mean missing data, absent data, or a structural limitation in the source.
